In [11]:
!pip install -U pageindex openai python-dotenv

Defaulting to user installation because normal site-packages is not writeable
  Using cached openai-3.41.0-py3-none-any.whl.metadata (44 kB)



[notice] A new release of pip is available: 26.1.2 -> 26.2.2
[notice] To update, run: C:\Python313\python.exe -m pip install --upgrade pip


In [12]:
import os, json, time
from dotenv import load_dotenv

load_dotenv()

# Load from environment — NO hardcoded APIs
PAGEINDEX_API_KEY = os.getenv("PAGEINDEX_API_KEY", "")
OPENAI_API_KEY    = os.getenv("OPENAI_API_KEY")

print("PageIndex key loaded:", "v" if PAGEINDEX_API_KEY else "Missing! Set in .env")
print("OpenAI key loaded:   ", "v" if OPENAI_API_KEY    else "Missing! Set in .env")

PageIndex key loaded: Missing! Set in .env
OpenAI key loaded:    v


In [13]:
from pageindex import PageIndexClient
from openai import OpenAI

pi_client     = PageIndexClient(api_key=PAGEINDEX_API_KEY)
openai_client = OpenAI(api_key=OPENAI_API_KEY)

print("PageIndex client ready")
print("OpenAI client ready")

ModuleNotFoundError: No module named 'pageindex'

In [14]:
PDF_PATH = "./sample_document.pdf"   # ← change this to your PDF

print(f"Uploading: {PDF_PATH}")
if PAGEINDEX_API_KEY:
    result = pi_client.submit_document(PDF_PATH)
    doc_id = result["doc_id"]
    print(f"Uploaded! Document ID: {doc_id}")
else:
    print("Skip upload — PAGEINDEX_API_KEY missing")

Uploading: ./sample_document.pdf
Skip upload — PAGEINDEX_API_KEY missing


In [15]:
print("Building tree index...")
print("   (Runs once per document — index cached for reuse)")

if PAGEINDEX_API_KEY and 'doc_id' in locals():
    while True:
        try:
            status_result = pi_client.get_document(doc_id)
            status = status_result.get("status")
            print(f"   Status: {status}")
            if status == "completed":
                print("\nTree index ready!")
                break
            elif status == "failed":
                print("\nProcessing failed. Check PDF format.")
                break
            time.sleep(5)
        except Exception as e:
            print(f"Status check error: {e}")
            time.sleep(5)
else:
    print("Skip — missing API key or doc_id")

Building tree index...
   (Runs once per document — index cached for reuse)
Skip — missing API key or doc_id


In [16]:
if PAGEINDEX_API_KEY and 'doc_id' in locals():
    try:
        tree_result = pi_client.get_tree(doc_id, node_summary=True)
        pageindex_tree = tree_result.get("result", [])
        print(f"📊 Top-level sections: {len(pageindex_tree)}")
        print("\nRaw tree (first node):")
        print(json.dumps(pageindex_tree[0] if pageindex_tree else {}, indent=2))
    except Exception as e:
        print(f"Tree fetch error: {e}")
else:
    print("Skip — set PAGEINDEX_API_KEY and upload first")

Skip — set PAGEINDEX_API_KEY and upload first


In [17]:
def print_tree(nodes, indent=0):
    for node in nodes:
        prefix = "  " * indent + ("└─ " if indent > 0 else "")
        page = node.get("page_index", "?")
        print(f"{prefix}[{node['node_id']}] {node['title']}  (p.{page})")
        if node.get("nodes"):
            print_tree(node["nodes"], indent + 1)

if PAGEINDEX_API_KEY and 'pageindex_tree' in locals():
    print("Full Document Structure:\n")
    print_tree(pageindex_tree)
else:
    print("Skip — tree not built")

Skip — tree not built


In [18]:
def count_nodes(nodes):
    total = len(nodes)
    for n in nodes:
        if n.get("nodes"):
            total += count_nodes(n["nodes"])
    return total

if PAGEINDEX_API_KEY and 'pageindex_tree' in locals():
    total = count_nodes(pageindex_tree)
    print(f"Total nodes in tree: {total}")
    print("   Each node = one retrievable section of the document")
else:
    print("Skip — tree not built")

Skip — tree not built


In [19]:
def llm_tree_search(query: str, tree: list, model: str = "gpt-4o") -> dict:
    def compress(nodes):
        out = []
        for n in nodes:
            entry = {
                "node_id": n["node_id"],
                "title":   n["title"],
                "page":    n.get("page_index", "?"),
                "summary": n.get("text", "")[:150]
            }
            if n.get("nodes"):
                entry["children"] = compress(n["nodes"])
            out.append(entry)
        return out
    
    compressed_tree = compress(tree)
    prompt = f"""You are given a query and a document tree (Table of Contents).
Identify which node IDs contain the answer.
Reply ONLY in JSON: {{\"thinking\": "...", \"node_list\": ["id1"]}}
Query: {query}

Document Tree:\n{json.dumps(compressed_tree, indent=2)}"""
    response = openai_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"}
    )
    return json.loads(response.choices[0].message.content)

if PAGEINDEX_API_KEY and OPENAI_API_KEY and 'pageindex_tree' in locals():
    query = "What is the syllabus covered in Modern LLM finetuning?"
    print(f"Query: {query}\n")
    result = llm_tree_search(query, pageindex_tree)
    print("LLM Reasoning:")
    print(result.get("thinking", "N/A"))
    print()
    print("Selected Node IDs:", result.get("node_list", []))
else:
    print("Skip — set both API keys and build tree first")

Skip — set both API keys and build tree first


In [1]:
def find_nodes_by_ids(tree: list, target_ids: list) -> list:
    found = []
    for node in tree:
        if node["node_id"] in target_ids:
            found.append(node)
        if node.get("nodes"):
            found.extend(find_nodes_by_ids(node["nodes"], target_ids))
    return found

def generate_answer(query: str, nodes: list, model: str = "gpt-4o") -> str:
    if not nodes:
        return "No relevant sections found in the document."
    context_parts = []
    for node in nodes:
        context_parts.append(
            f"[Section: '{node['title']}' | Page {node.get('page_index', '?')}]\n"
            f"{node.get('text', 'Content not available.')}"
        )
    context = "\n\n---\n\n".join(context_parts)
    prompt = f"""You are an expert document analyst.
Answer using ONLY context. Cite section titles and page numbers.
Question: {query}\n\nContext:\n{context}\n\nAnswer:"""
    response = openai_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

if PAGEINDEX_API_KEY and OPENAI_API_KEY and 'pageindex_tree' in locals():
    selected_ids = result.get("node_list", []) if 'result' in locals() else []
    nodes = find_nodes_by_ids(pageindex_tree, selected_ids)
    answer = generate_answer(query, nodes)
    print("\n=== ANSURED ANSWER ===")
    print(answer)
else:
    print("Skip — complete upload + tree + search first")

NameError: name 'PAGEINDEX_API_KEY' is not defined

In [2]:
def vectorless_rag(query: str, tree: list, verbose: bool = True) -> str:
    """
    Full end-to-end PageIndex RAG pipeline:
    
    Step 1: LLM Tree Search  → finds relevant node_ids
    Step 2: Node Retrieval   → fetches section content
    Step 3: Answer Generation → produces cited answer
    """
    if verbose:
        print(f"{'='*55}")
        print(f"Query: {query}")
        print(f"{'='*55}")
    
    # Step 1: Tree Search
    search_result  = llm_tree_search(query, tree)
    node_ids       = search_result.get("node_list", [])
    
    if verbose:
        print(f"\nReasoning: {search_result.get('thinking', '')[:200]}...")
        print(f"Retrieved node IDs: {node_ids}")
    
    # Step 2: Retrieve nodes
    nodes = find_nodes_by_ids(tree, node_ids)
    
    if verbose:
        print(f"Sections found: {[n['title'] for n in nodes]}")
    
    # Step 3: Generate answer
    answer = generate_answer(query, nodes)
    
    if verbose:
        print(f"\nAnswer:\n{answer}")
    
    return answer

In [3]:
answer = vectorless_rag(
    query="What are the syllabus covered in modern llm finetuning?",
    tree=pageindex_tree
)

NameError: name 'pageindex_tree' is not defined

In [4]:
test_queries = [
    "What are the syllabus covered in modern llm finetuning?",
    "What are the syllabus covered in RAG?",
    "Summarize the syllabus of Tokenization Deep Dive?",
]

for q in test_queries:
    print()
    ans = vectorless_rag(q, pageindex_tree, verbose=False)
    print(f"Q: {q}")
    print(f"A: {ans[:300]}...")
    print("-" * 55)

NameError: name 'pageindex_tree' is not defined

In [5]:
FINANCIAL_EXPERT_RULES = """
Expert routing rules for financial documents (10-K, annual reports):
- EBITDA, profitability queries    → MD&A section (Management Discussion & Analysis)
- Liquidity, cash flow queries     → Cash Flow Statement + liquidity footnotes
- Risk factor queries              → Part I, Item 1A (Risk Factors)  
- Revenue breakdown queries        → Segment reporting or Item 7
- Forward-looking / strategy       → CEO letter, Outlook, Strategy section
- Debt, credit, leverage queries   → Balance Sheet + debt footnotes
- Regulatory / compliance queries  → Legal Proceedings or regulatory filings
"""

print("✅ Expert rules defined")
print("   These get injected into the retrieval prompt at query time.")

✅ Expert rules defined
   These get injected into the retrieval prompt at query time.


In [ ]:
FINANCIAL_EXPERT_RULES = """
Route queries to the correct module using these rules:
 
M1  Neural Network Refresher   → backprop, activations, optimizers, PyTorch basics
M2  Hardware                   → GPU, TPU, Apple Silicon, compute infrastructure
M3  Transformers 101           → attention, self-attention, encoder-decoder, MHA
M4  Tokenization               → BPE, WordPiece, SentencePiece, Byte Latent Transformers
M5  Finetuning Architectures   → hands-on BERT/GPT/T5 finetuning, Hugging Face
M6  KV Cache & Attention       → KV cache, Flash Attention, MQA, GQA, RoPE, vLLM
M7  Scaling Laws               → Kaplan, Chinchilla, compute-optimal training
M8  Mixture of Experts         → MoE, sparse computation, Mixture of Depths
M9  Modern LLM Finetuning      → LoRA, QLoRA, SFT, DPO, PPO, RLHF, GRPO, ORPO,
                                  quantization, TRL, Unsloth, synthetic data,
                                  reasoning models, evaluation, deployment
M10 SLM                        → small language models, pruning, when SLM vs LLM
M11 Knowledge Distillation     → student-teacher, soft labels, DistilBERT, DeepSeek-R1
M12 Hybrid Architectures       → Mamba, RWKV, SSMs, Jamba, Nemotron, beyond Transformers
M13 Vision Foundations         → ViT, patch embeddings, CLIP, SigLIP, DINOv2
M14 Visual Language Models     → VLM architecture, aligner, multimodal reasoning
M15 Stable Diffusion & DiT     → DDPM, latent diffusion, FLUX.1, ControlNet, DreamBooth
M16 Embedding Models           → dense, sparse, binary, Matryoshka, MRL, fine-tuning
M17 RAG                        → chunking, BM25, ColBERT, hybrid RAG, rerankers,
                                  self/corrective/adaptive/agentic RAG, Graph RAG,
                                  multi-modal RAG, ColPali, RAG security
M18 Context Engineering        → prompt vs context engineering, memory architecture,
                                  context compression, KV cache, agent context lifecycle
M19 DSPy                       → signatures, modules, MIPROv2, self-optimizing RAG
M20 Agents                     → ReAct, MCP, LangGraph, CrewAI, browser agents,
                                  A2A, guardrails, observability, evaluation
M21 RL                         → PPO, GRPO, DAPO, GSPO, CISPO, reward models,
                                  RLHF vs RLVR, policy gradient, DeepSeek-R1 training
 
Cross-cutting rules:
- "learning path / where to start"     → M1 → M2 → M3 in order
- "production / deployment / serving"  → M9 (quantization) + M20 (agents)
- "fine-tuning vs RAG"                 → M9 + M17 + M18
- "multimodal / vision + language"     → M13 + M14 + M17 (multi-modal RAG)
- "reasoning models / test-time RL"    → M9 (reasoning) + M21 (GRPO/DAPO)
"""